In [10]:
import numpy as np
import pandas as pd
import seaborn as sns
from shapely import wkt
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import matplotlib.patches as mpatches
from shapely.geometry import MultiPoint
from sklearn.neighbors import NearestNeighbors

# Set global display format to show up to 6 decimal places
pd.options.display.float_format = '{:.6f}'.format

BASE_DIR = Path('/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/')

# Load Waste Data

In [26]:
df = pd.read_csv(BASE_DIR/"img/Combined_SVI.csv") # all
# df = pd.read_csv(BASE_DIR/"img/Correct_SVI.csv") # only identified
# df = df[~df['img_dir'].isin(['Faith/', 'ZWL/'])]
# df[df['Domestic'] == 'Y']
df

/var/folders/2_/nk9j6sb901n_fk5dz_9vtqj80000gn/T/ipykernel_42574/4294929812.py:1: DtypeWarning: Columns (14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(BASE_DIR/"img/Combined_SVI.csv") # all


,img_name,year,month,day,hour,lat,lon,panoid,img_dir,exist,yolo_conf,yolo_bbox,yolo_num,prediction,Domestic,Construction
0,014652c8e7e36740a0fe390870e19052b70a2e9a40,2015,8,22.000000,9.000000,-1.268261,36.821169,NaN,Faith/,True,[],[],0,pending,NaN,NaN
1,20161115_133906,2016,11,15.000000,13.000000,-1.317033,36.791557,NaN,Faith/,True,"[0.4858177900314331, 0.3025636374950409]","[[28.5167236328125, 1253.941650390625, 776.544...",2,yes,Y,N
2,IMG_1450,2016,11,14.000000,13.000000,-1.289936,36.769103,NaN,Faith/,True,[],[],0,pending,NaN,NaN
3,IMG_1336,2016,11,14.000000,11.000000,-1.303164,36.855931,NaN,Faith/,True,[],[],0,pending,NaN,NaN
4,20161115_134714,2016,11,15.000000,13.000000,-1.315711,36.790286,NaN,Faith/,True,[],[],0,pending,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
457106,IMG_0794,2023,2,20.000000,11.000000,-1.361619,36.763569,NaN,ZWL/,True,[],[],0,pending,NaN,NaN
457107,IMG_20230211_130003,2023,2,11.000000,13.000000,NaN,NaN,NaN,ZWL/,True,[],[],0,pending,NaN,NaN
457108,IMG_8486,2023,2,11.000000,11.000000,-1.278647,37.038514,NaN,ZWL/,True,[],[],0,pending,NaN,NaN
457109,IMG_8492,2023,2,11.000000,11.000000,-1.278183,37.033633,NaN,ZWL/,True,[],[],0,pending,NaN,NaN


In [27]:
# Convert to a GeoDataFrame
df['geometry'] = df.apply(lambda row: Point(row['lon'], row['lat']), axis=1)
gdf_waste = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")
gdf_waste

,img_name,year,month,day,hour,lat,lon,panoid,img_dir,exist,yolo_conf,yolo_bbox,yolo_num,prediction,Domestic,Construction,geometry
0,014652c8e7e36740a0fe390870e19052b70a2e9a40,2015,8,22.000000,9.000000,-1.268261,36.821169,NaN,Faith/,True,[],[],0,pending,NaN,NaN,POINT (36.82117 -1.26826)
1,20161115_133906,2016,11,15.000000,13.000000,-1.317033,36.791557,NaN,Faith/,True,"[0.4858177900314331, 0.3025636374950409]","[[28.5167236328125, 1253.941650390625, 776.544...",2,yes,Y,N,POINT (36.79156 -1.31703)
2,IMG_1450,2016,11,14.000000,13.000000,-1.289936,36.769103,NaN,Faith/,True,[],[],0,pending,NaN,NaN,POINT (36.7691 -1.28994)
3,IMG_1336,2016,11,14.000000,11.000000,-1.303164,36.855931,NaN,Faith/,True,[],[],0,pending,NaN,NaN,POINT (36.85593 -1.30316)
4,20161115_134714,2016,11,15.000000,13.000000,-1.315711,36.790286,NaN,Faith/,True,[],[],0,pending,NaN,NaN,POINT (36.79029 -1.31571)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
457106,IMG_0794,2023,2,20.000000,11.000000,-1.361619,36.763569,NaN,ZWL/,True,[],[],0,pending,NaN,NaN,POINT (36.76357 -1.36162)
457107,IMG_20230211_130003,2023,2,11.000000,13.000000,NaN,NaN,NaN,ZWL/,True,[],[],0,pending,NaN,NaN,POINT (NaN NaN)
457108,IMG_8486,2023,2,11.000000,11.000000,-1.278647,37.038514,NaN,ZWL/,True,[],[],0,pending,NaN,NaN,POINT (37.03851 -1.27865)
457109,IMG_8492,2023,2,11.000000,11.000000,-1.278183,37.033633,NaN,ZWL/,True,[],[],0,pending,NaN,NaN,POINT (37.03363 -1.27818)


# Load Social Economic data

In [28]:
df_SE = pd.read_csv(BASE_DIR/"SE/ken_2023_admin3_nairobi_mombasa_v3.csv")
# df_SE
df_SE['geometry'] = df_SE['geometry'].apply(wkt.loads)
gdf_SE = gpd.GeoDataFrame(df_SE, geometry='geometry')
gdf_SE.set_crs(epsg=4326, inplace=True)
gdf_SE

,target,geometry,County,SubCounty,Division,Location,SubLocation,source_id,devices,tests,...,density_GSM,density_UMTS,density_LTE,density_tests,density_devices,density_medical,density_young,density_young_city,density_old,density_old_city
0,0.763519,"POLYGON ((36.81612 -1.27604, 36.8165 -1.27626,...",Nairobi,Starehe,Cbd,47100103 - Starehe,4710010301 - City Centre,1,353,563,...,3604.832291,148.195317,210.632939,68579.633011,42999.308087,487.244284,1577.533710,6054.276894,40.935995,132.883030
1,0.820513,"POLYGON ((36.82276 -1.29479, 36.82202 -1.29521...",Nairobi,Starehe,Cbd,47100103 - Starehe,4710010302 - City Square,2,213,459,...,3714.519512,202.752488,214.494910,100866.106286,46807.147361,1098.759328,853.601422,6054.276894,28.555656,132.883030
2,0.616973,"POLYGON ((36.84376 -1.2673, 36.84544 -1.27617,...",Nairobi,Starehe,Central,47100202 - Pangani,4710020201 - Pangani,3,64,169,...,4302.526131,163.374685,185.875390,5396.070710,2043.482399,127.717650,16529.922800,6054.276894,407.419851,132.883030
3,0.539104,"POLYGON ((36.84544 -1.27617, 36.84622 -1.28015...",Nairobi,Starehe,Central,47100201 - Kariokor,NaN,4,55,92,...,3444.603569,139.248404,150.732809,2673.810441,1598.473633,87.189471,13997.876050,6054.276894,344.536969,132.883030
4,0.296127,"POLYGON ((36.85074 -1.25821, 36.8502 -1.26025,...",Nairobi,Mathare,Mathare,47080201 - Mathare,4708020102 - Mathare,5,21,32,...,5461.526458,189.813959,226.161313,784.336195,514.720628,0.000000,18249.713018,20877.654239,452.335655,514.183431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,0.377485,"POLYGON ((39.60523 -4.00982, 39.60525 -4.00983...",Mombasa,Changamwe,Changamwe,01010104 - Port Reitz,0101010401 - Port Reitz,1147,70,174,...,238.051577,30.940391,0.000000,2612.426161,1050.976042,75.069717,2338.272550,2498.025469,64.496477,69.410605
312,0.414160,"POLYGON ((39.61748 -4.00857, 39.6172 -4.00852,...",Mombasa,Jomvu,Jomvu,01020101 - Mikindani,0102010102 - Kwa Shee,1148,26,48,...,484.418180,66.022922,0.000000,1537.500357,832.812694,96.093772,3456.150022,1593.491700,95.697679,44.910399
313,0.267615,"POLYGON ((39.64012 -4.03077, 39.63913 -4.03072...",Mombasa,Jomvu,Jomvu,01020101 - Mikindani,0102010101 - Birikani,1149,4,5,...,1135.045890,114.035939,0.408731,278.845499,223.076399,0.000000,3147.117106,1593.491700,87.367830,44.910399
314,0.263161,"POLYGON ((39.60284 -4.01424, 39.60271 -4.01447...",Mombasa,Jomvu,Jomvu,01020102 - Miritini,0102010202 - Miritini,1150,17,25,...,201.056159,29.494241,0.000000,1097.830001,746.524401,131.739600,964.981659,1593.491700,27.055466,44.910399


# Match
some data is missing cause IDEAMaps has a larger boundary

In [29]:
# Spatial join: assign each image to a boundary polygon
gdf_joined = gpd.sjoin(gdf_waste, gdf_SE, how="inner", predicate="within")
gdf_joined

,img_name,year,month,day,hour,lat,lon,panoid,img_dir,exist,...,density_GSM,density_UMTS,density_LTE,density_tests,density_devices,density_medical,density_young,density_young_city,density_old,density_old_city
0,014652c8e7e36740a0fe390870e19052b70a2e9a40,2015,8,22.000000,9.000000,-1.268261,36.821169,NaN,Faith/,True,...,1124.691000,36.031920,82.515847,20437.909539,12791.863723,29.072418,2245.767125,1469.699832,168.442973,109.831736
1,20161115_133906,2016,11,15.000000,13.000000,-1.317033,36.791557,NaN,Faith/,True,...,9926.882640,598.784900,521.104697,706.301186,431.628502,0.000000,13164.792021,9646.946755,396.497093,308.305583
2,IMG_1450,2016,11,14.000000,13.000000,-1.289936,36.769103,NaN,Faith/,True,...,612.890311,28.341748,44.071419,15104.413482,8730.874845,0.000000,1458.120120,1933.441698,91.768145,129.121861
3,IMG_1336,2016,11,14.000000,11.000000,-1.303164,36.855931,NaN,Faith/,True,...,1024.451244,80.923331,9.877152,1419.027254,644.097477,60.384138,7945.981194,7887.047949,139.122282,152.820910
4,20161115_134714,2016,11,15.000000,13.000000,-1.315711,36.790286,NaN,Faith/,True,...,9926.882640,598.784900,521.104697,706.301186,431.628502,0.000000,13164.792021,9646.946755,396.497093,308.305583
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
457103,IMG_6497,2023,2,3.000000,13.000000,-1.306239,36.786778,NaN,ZWL/,True,...,4051.506133,229.463751,234.170700,862.780963,496.099054,194.125717,28448.718077,9646.946755,856.818212,308.305583
457104,IMG_6483,2023,2,3.000000,13.000000,-1.305792,36.785050,NaN,ZWL/,True,...,1030.054347,54.580106,64.102593,2491.053915,1372.190716,63.331879,4057.518567,9646.946755,145.304587,308.305583
457106,IMG_0794,2023,2,20.000000,11.000000,-1.361619,36.763569,NaN,ZWL/,True,...,63.444064,6.337019,1.157409,4573.814216,1742.405415,93.343147,80.927872,364.582738,2.927817,11.442383
457108,IMG_8486,2023,2,11.000000,11.000000,-1.278647,37.038514,NaN,ZWL/,True,...,10.473427,0.815895,0.149858,6528.232777,3766.288141,188.314407,96.391038,832.119698,2.009246,17.301924


# Summary of Metrics

| Metric                    | Purpose                                   |
| ------------------------- | ----------------------------------------- |
| `waste_pile_count`        | Raw count (biased by sampling effort)     |
| `waste_per_image`         | Adjusts for sampling effort               |
| `waste_density_per_km2`   | Adjusts for area size                     |
| `waste_per_image_per_km2` | Adjusts for both sampling effort and area |


## Step 1: Metric 1 — Count of Waste Piles (Domestic == 'Y')

In [30]:
# Filter for actual waste detections
gdf_detected = gdf_joined[gdf_joined['Domestic'] == 'Y']

# Count domestic waste detections per admin unit
waste_counts = gdf_detected.groupby('source_id').size().reset_index(name='waste_pile_count')

## Step 2: Metric 2 — Count of All Images per Admin Unit (Sampling Effort)

In [31]:
# Count unique images per admin unit (sampling effort)
image_counts = gdf_joined.groupby('source_id')['img_name'].nunique().reset_index(name='image_count')
image_counts

,source_id,image_count
0,1,3084
1,2,2492
2,3,1524
3,4,1212
4,5,500
...,...,...
160,220,164
161,221,524
162,735,22544
163,736,2220


## Step 3: Metric 3 — Admin Area in km²

In [32]:
# Convert to projected CRS for area calculation
gdf_SE_proj = gdf_SE.to_crs(epsg=3395)
gdf_SE_proj['area_km2'] = gdf_SE_proj.geometry.area / 10**6

# Extract area for merging
area_df = gdf_SE_proj[['source_id', 'area_km2']]
area_df

,source_id,area_km2
0,1,1.329988
1,2,1.278060
2,3,1.022688
3,4,1.393884
4,5,0.743191
...,...,...
311,1147,9.549018
312,1148,4.069902
313,1149,2.485332
314,1150,8.926574


## Step 4: Merge All Metrics

In [33]:
# Merge counts
summary = pd.merge(image_counts, waste_counts, on='source_id', how='left')
summary = pd.merge(summary, area_df, on='source_id', how='left')

# Replace NaNs (e.g., if no waste found in a unit)
summary['waste_pile_count'] = summary['waste_pile_count'].fillna(0)
summary

,source_id,image_count,waste_pile_count,area_km2
0,1,3084,37.000000,1.329988
1,2,2492,2.000000,1.278060
2,3,1524,53.000000,1.022688
3,4,1212,20.000000,1.393884
4,5,500,32.000000,0.743191
...,...,...,...,...
160,220,164,0.000000,5.270685
161,221,524,0.000000,5.387967
162,735,22544,222.000000,143.926823
163,736,2220,7.000000,66.114311


## Step 5: Compute Final Metrics

In [34]:
# Metric 2: Waste per image
summary['waste_per_image'] = summary.apply(
    lambda row: row['waste_pile_count'] / row['image_count'] if row['image_count'] > 0 else 0,
    axis=1
)

# Metric 3: Waste density per km²
summary['waste_density_per_km2'] = summary.apply(
    lambda row: row['waste_pile_count'] / row['area_km2'] if row['area_km2'] > 0 else 0,
    axis=1
)

# Metric 4: Waste per image per km²
summary['waste_per_image_per_km2'] = summary.apply(
    lambda row: row['waste_per_image'] / row['area_km2'] if row['area_km2'] > 0 else 0,
    axis=1
)
summary

,source_id,image_count,waste_pile_count,area_km2,waste_per_image,waste_density_per_km2,waste_per_image_per_km2
0,1,3084,37.000000,1.329988,0.011997,27.819800,0.009021
1,2,2492,2.000000,1.278060,0.000803,1.564871,0.000628
2,3,1524,53.000000,1.022688,0.034777,51.824197,0.034005
3,4,1212,20.000000,1.393884,0.016502,14.348395,0.011839
4,5,500,32.000000,0.743191,0.064000,43.057566,0.086115
...,...,...,...,...,...,...,...
160,220,164,0.000000,5.270685,0.000000,0.000000,0.000000
161,221,524,0.000000,5.387967,0.000000,0.000000,0.000000
162,735,22544,222.000000,143.926823,0.009847,1.542450,0.000068
163,736,2220,7.000000,66.114311,0.003153,0.105877,0.000048


In [35]:
# Merge Back to GeoDataFrame for Mapping
gdf_SE_summary = gdf_SE.merge(summary, on='source_id', how='left')
gdf_SE_summary

,target,geometry,County,SubCounty,Division,Location,SubLocation,source_id,devices,tests,...,density_young,density_young_city,density_old,density_old_city,image_count,waste_pile_count,area_km2,waste_per_image,waste_density_per_km2,waste_per_image_per_km2
0,0.763519,"POLYGON ((36.81612 -1.27604, 36.8165 -1.27626,...",Nairobi,Starehe,Cbd,47100103 - Starehe,4710010301 - City Centre,1,353,563,...,1577.533710,6054.276894,40.935995,132.883030,3084.000000,37.000000,1.329988,0.011997,27.819800,0.009021
1,0.820513,"POLYGON ((36.82276 -1.29479, 36.82202 -1.29521...",Nairobi,Starehe,Cbd,47100103 - Starehe,4710010302 - City Square,2,213,459,...,853.601422,6054.276894,28.555656,132.883030,2492.000000,2.000000,1.278060,0.000803,1.564871,0.000628
2,0.616973,"POLYGON ((36.84376 -1.2673, 36.84544 -1.27617,...",Nairobi,Starehe,Central,47100202 - Pangani,4710020201 - Pangani,3,64,169,...,16529.922800,6054.276894,407.419851,132.883030,1524.000000,53.000000,1.022688,0.034777,51.824197,0.034005
3,0.539104,"POLYGON ((36.84544 -1.27617, 36.84622 -1.28015...",Nairobi,Starehe,Central,47100201 - Kariokor,NaN,4,55,92,...,13997.876050,6054.276894,344.536969,132.883030,1212.000000,20.000000,1.393884,0.016502,14.348395,0.011839
4,0.296127,"POLYGON ((36.85074 -1.25821, 36.8502 -1.26025,...",Nairobi,Mathare,Mathare,47080201 - Mathare,4708020102 - Mathare,5,21,32,...,18249.713018,20877.654239,452.335655,514.183431,500.000000,32.000000,0.743191,0.064000,43.057566,0.086115
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,0.377485,"POLYGON ((39.60523 -4.00982, 39.60525 -4.00983...",Mombasa,Changamwe,Changamwe,01010104 - Port Reitz,0101010401 - Port Reitz,1147,70,174,...,2338.272550,2498.025469,64.496477,69.410605,NaN,NaN,NaN,NaN,NaN,NaN
312,0.414160,"POLYGON ((39.61748 -4.00857, 39.6172 -4.00852,...",Mombasa,Jomvu,Jomvu,01020101 - Mikindani,0102010102 - Kwa Shee,1148,26,48,...,3456.150022,1593.491700,95.697679,44.910399,NaN,NaN,NaN,NaN,NaN,NaN
313,0.267615,"POLYGON ((39.64012 -4.03077, 39.63913 -4.03072...",Mombasa,Jomvu,Jomvu,01020101 - Mikindani,0102010101 - Birikani,1149,4,5,...,3147.117106,1593.491700,87.367830,44.910399,NaN,NaN,NaN,NaN,NaN,NaN
314,0.263161,"POLYGON ((39.60284 -4.01424, 39.60271 -4.01447...",Mombasa,Jomvu,Jomvu,01020102 - Miritini,0102010202 - Miritini,1150,17,25,...,964.981659,1593.491700,27.055466,44.910399,NaN,NaN,NaN,NaN,NaN,NaN
